In [ ]:
#libraries
from bs4 import BeautifulSoup, element
import urllib
import requests
import pandas as pd
import numpy as np
import lyricsgenius
import re
import json

#NLP stuff
#Tokenization
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords #look into if I should keep this or not
from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

from gensim import corpora
from gensim.models.ldamodel import LdaModel

from bertopic import BERTopic

# Functions

In [ ]:
# ── Functions for Song Searching and Processing ────────────────────────────────────────────────
def clean_lyrics(raw_lyrics):
    """
    Strips metadata and standardizes raw Genius lyrics into clean plain text.
    This function removes that metadata along with section headers, extra
    whitespace, punctuation, and capitalisation.

    Args:
        raw_lyrics (str): Raw lyrics string returned by genius.search_song()

    Returns:
        str: Cleaned, lowercased lyrics ready for tokenization
    """
    
    # Genius prepends metadata before the first section header e.g. [Intro]
    starter = ']' # Finding the first ']' and slicing from there drops all prepended metadata
    position = raw_lyrics.find(starter)
    lyrics = raw_lyrics[position+1:]

    # Remove section headers like [Chorus], [Verse 1], etc.
    lyrics = re.sub(r"\[.*?\]", "", lyrics)

    # Remove extra whitespace, newlines
    lyrics = re.sub(r"\n+", " ", lyrics)
    lyrics = re.sub(r"\s+", " ", lyrics)

    # Remove punctuation
    lyrics = re.sub(r"[^\w\s']", "", lyrics)

    # Lowercase
    lyrics = lyrics.lower()

    return lyrics


def song_search(song,artist):
    """
    Fetches and cleans lyrics for a given song from the Genius API.

    Initialises a Genius API client, searches for the song by name and
    artist, then passes the raw lyrics through clean_lyrics() before
    returning them.

    Args:
        song (str):   Title of the song to search for
        artist (str): Name of the artist who recorded the song

    Returns:
        str: Cleaned lyrics string
    """

    genius = lyricsgenius.Genius(genius_token, timeout=60)
    song = genius.search_song(song, artist)
    lyrics = song.lyrics

    # Handles if song is not found
    if song is None:
        print(f"Could not find lyrics for '{song}' by {artist}")
        return None
    
    cleaned_lyrics = clean_lyrics(lyrics)
    return cleaned_lyrics

In [ ]:
df = pd.read_csv(r'C:\Users\Monado\Documents\Data_git\Projects\Lyrics_NLP\english_song_dataset.csv')
print(df)

In [ ]:
## For loop for adding songs to list 
for i in range(67,len(songs)):
    cleaned_lyrics = song_search(songs[i], artists[i])
    with open("lyrics_list.txt", "a", encoding="utf-8") as file:
        file.write("\n" + cleaned_lyrics + ',' )

In [ ]:
#appends lyrics to txt file to store for later use with NLP model
with open("lyrics_list.txt", "a", encoding="utf-8") as file:
    file.write("\n" + cleaned_lyrics + ',' )

In [ ]:
#reads the list of lyrics from the txt file
with open("lyrics_list.txt", "r", encoding="utf-8") as f:
    text = f.read()
    lyrics_list = [song for song in text.split(',') if song]